In [34]:
# manipulacja danymi
import numpy as np
import pandas as pd

# wizualizacja
import matplotlib.pyplot as plt
import seaborn as sns

# podział danych na zbiory treningowe/walidacyjne/testowe
from sklearn.model_selection import train_test_split, GridSearchCV

# budowa Pipeline
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer

# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures, PowerTransformer

# redukcja wymiarowości
from sklearn.decomposition import PCA

# model
from sklearn.linear_model import LinearRegression

# ewaluacja
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, f1_score, roc_auc_score

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [35]:
dataset = pd.read_csv('daily-bike-share.csv')

In [36]:
dataset.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,rentals
0,1,1/1/2011,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331
1,2,1/2/2011,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131
2,3,1/3/2011,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120
3,4,1/4/2011,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108
4,5,1/5/2011,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82


In [37]:
dataset.iloc[0]

instant              1
dteday        1/1/2011
season               1
yr                   0
mnth                 1
holiday              0
weekday              6
workingday           0
weathersit           2
temp          0.344167
atemp         0.363625
hum           0.805833
windspeed     0.160446
rentals            331
Name: 0, dtype: object

In [38]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    object 
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  rentals     731 non-null    int64  
dtypes: float64(4), int64(9), object(1)
memory usage: 80.1+ KB


In [55]:
X = dataset.drop(['rentals','instant'],axis=1)
y = dataset['rentals']

In [56]:
#train/test split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

In [57]:
# =========================
# PODZIAŁ KOLUMN
# =========================

# zmienne numeryczne
num_features = [
    'temp',
    'atemp',
    'hum',
    'windspeed'
]

# zmienne kategoryczne
cat_features = [
    'season',
    'yr',
    'mnth',
    'holiday',
    'weekday',
    'workingday',
    'weathersit'
]

In [71]:
# =========================
# PRZYGOTOWANIE DANYCH NUMERYCZNYCH
# =========================

num_preparation = Pipeline([
    ('scaler', StandardScaler())
])

In [72]:
# =========================
# PRZYGOTOWANIE DANYCH KATEGORYCZNYCH
# =========================

cat_preparation = Pipeline(steps=[

    # zamiana kategorii na kolumny binarne (0/1)
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))

])

In [73]:
# =========================
# COLUMN TRANSFORMER
# =========================

data_preparation = ColumnTransformer(
    transformers=[
        ('num', num_preparation, num_features),
        ('cat', cat_preparation, cat_features)
    ],
    remainder='drop'
)

In [74]:
# =========================
# PEŁNY PIPELINE
# =========================

model_pipeline = Pipeline(steps=[

    # przygotowanie danych
    ('preprocessor', data_preparation),

    # model regresji liniowej
    ('model', LinearRegression())

])

In [75]:
model_pipeline.fit(X_train, y_train)

y_predict_train = model_pipeline.predict(X_train)
y_predict_test = model_pipeline.predict(X_test)

In [76]:
print(f'R2: {r2_score(y_test, y_predict_test)}')
print(f'MAE: {mean_absolute_error(y_test, y_predict_test)}')
print(f'MSE: {mean_squared_error(y_test, y_predict_test)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_predict_test))}')

R2: 0.6977308202777837
MAE: 238.8714118490162
MSE: 104446.63231005959
RMSE: 323.1820420599814


In [77]:
print('TRAIN')
print(f'R²: {r2_score(y_train, y_predict_train):.4f}')
print(f'MAE: {mean_absolute_error(y_train, y_predict_train):.4f}')
print(f'MSE: {mean_squared_error(y_train, y_predict_train):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_train, y_predict_train)):.4f}')

print('\nTEST')
print(f'R²: {r2_score(y_test, y_predict_test):.4f}')
print(f'MAE: {mean_absolute_error(y_test, y_predict_test):.4f}')
print(f'MSE: {mean_squared_error(y_test, y_predict_test):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_predict_test)):.4f}')

TRAIN
R²: 0.7479
MAE: 262.0252
MSE: 127985.4011
RMSE: 357.7505

TEST
R²: 0.6977
MAE: 238.8714
MSE: 104446.6323
RMSE: 323.1820


# WYNIKI Z MODULU 12

  Metryka  Model końcowy
0      R²       0.524682
1     MAE     348.585194
2     MSE  234195.931639
3    RMSE     483.937942

# WNIOSKI

W ramach zadania zbudowano Pipeline automatyzujący proces przygotowania danych oraz trenowania modelu regresji liniowej. Dla zmiennych numerycznych zastosowano standaryzację, natomiast zmienne kategoryczne zostały zakodowane przy użyciu One-Hot Encoding. Następnie wykorzystano model LinearRegression do przewidywania liczby wypożyczeń rowerów.

Przeprowadzono również eksperyment z wykorzystaniem PolynomialFeatures, jednak spowodował on przeuczenie modelu. Pomimo wysokiego wyniku na zbiorze treningowym, jakość predykcji na zbiorze testowym znacząco spadła, co świadczyło o słabej zdolności generalizacji.

Ostateczny Pipeline bez cech wielomianowych osiągnął współczynnik determinacji R² równy 0.6977 na zbiorze testowym, co stanowi znaczącą poprawę względem wcześniejszego modelu, który uzyskał wynik R² równy 0.5247. Dodatkowo zmniejszeniu uległy błędy MAE, MSE oraz RMSE, co potwierdza wyższą jakość predykcji. Zastosowanie Pipeline pozwoliło uporządkować proces przygotowania danych, ograniczyć ryzyko błędów oraz ułatwić dalszą rozbudowę modelu.
